

c) Consider how you should post-process the output of the model to provide metrics thatare suitable for the shared task. In particular, you must ensure that the number oftokens in your confusion matrix match the number of predictions expected by the test set. This needs to be motivated and dealt with explicitly.

d) Prepare the code to evaluate your model on the evaluation set. Provide Precision, Recall and F1 measures for token-level classification. Make sure to also include a labeled confusion matrix that supports the classification metrics. Store your model’s predictions over the preprocessed test set and save it in text format (e.g., as a tsv). Make sure this file contains, at least, the token, the gold label and the predicted label for each prediction.

e) Prepare a ready-to-use function where one can use the trained model to perform SRL on standalone sentences, given the predicate. Among other necessary
arguments (e.g., model, etc.), the function should allow the input of a sentence
segmented as a list of strings (e.g., [‘Pia’, ‘asked’, ‘Luis’, ‘to’, ‘write’, ‘this’, ‘sentence’,’.’]) and a list defining the location of the predicate to label (e.g. [0,0,0,0,1,0,0,0], for the predicate ‘write’). If you think of another better way to design this function, that is also acceptable, as long as it is well documented. Provide an example showing that the function runs a sentence with more than one predicate (you can choose your own
sentence(s)!). Note: this function will be important for your take-home exam.



# Fine-tuning a model on a token classification task

notebook adapted from: https://github.com/huggingface/notebooks/blob/main/examples/token_classification.ipynb


In this notebook, we will fine-tune a BERT model. Specifically, we will apply it to **Named Entity Recognition** as in our previous assignments.

Your task is to understand how we can do this by following the notebook and

1.   understand how we can do this by following the notebook and getting familiar with the HuggingFace ecosystem.
2.   fine-tuning the BERT model with 3 different random seeds and reporting the results in the report.

We will see how to easily load a dataset for this task and use the HuggingFace `Trainer` API to fine-tune a model on it.

Let's install the packages we need from 🤗 : Transformers, Datasets, Seqeval, and Evalaute.

In [1]:
! pip install transformers seqeval evaluate

In [2]:

model_checkpoint = "google-bert/bert-base-uncased"


## Loading the dataset

The following three functions respectively extract the information from the files, preprocess it and write it to preprocessed files, and extracting the info of these new files.

In [3]:

def extract_tokens_and_arg_info(conllfile):
    """
    Reads a Conll file and extracts the sentences and tokens; where each sentence is made of tokens which are tupled with
    argument information.
    Args:
        inputfile: a file in the Conllu format
    Returns:
        A list of sentences represented by a list of tokens, which are structured as tuples with their argument information."""

    allsentences = []
    currentsentence=[]
    with open(conllfile, 'r', encoding='utf8') as infile:
        for line in infile:
            #turn lines into strings
            line = line.strip()
            #sentence boundaries
            if line == '':
            #if boundary is reached and current sentence is full, append it to the larger list
                if currentsentence:
                    allsentences.append(currentsentence)
                    #restarts current sentence
                    currentsentence = []
            #if we are not at a boundary
            else:
                #if we are not at a metadata line
                if not line.startswith('#'):
                    #finds information per line
                    components = line.rstrip('\n').split()
                    index=components[0]
                    #if there are predicates in the sentence, since there would be more than 10 columns
                    #and excluding copies which have a dot in their index
                    if len(components)>10 and '.' not in index:
                        #for each columns of arguments, which starts from the tenth(indexing including id
                        # and since index range is inclusive ) till the nth one
                        token=[components[1]]
                        for component in components[11:]:
                            #appends argument info to token
                            token.append(component)
                        #appends tuppled token information to sentence
                        currentsentence.append(tuple(token))


    return  allsentences


In [4]:
import csv

def data_preprocessing(extracteddata,outputfile):
    ''' Preprocesses the data extracted from the Conllu files, multiples sentences with multiple predicates in the dataset,
    finds the verb, and the gold labels per predicate.
    Args:
        inputfile: output of the function extract_tokens_and_arg_info(); or data in the following format: list of lists(sentences)
        of tuples(tokens) containing the token and all its labels of arguments in that sentence. [[(token, arginfo,+)]]

    Returns: list of lists-sentences with respectives repetitions according to number of verbs,
        list of lists- goldaarguments lists, one per sentence/repeated sentence
        list- the token of the predicate corresponding to the list of gold labels'''

    verbs=[]
    tokenonlysentences=[]
    goldarguments=[]
    sentence_lenght=[]

    for sentence in extracteddata:
        #to further avoid cases with no verbs, checks for the lenght
        if len(sentence[0])<2:
            continue
        else:
            tokensentence=[]
            #finds number of verbs through lenght of tuple per sentence
            numofverbs=len(sentence[0])-1

            if len(sentence)>1:

            #starts sorting the argument information of each token into separate tuples
                sortedargumentinfo = [list(pair) for pair in zip(sentence[0][1:], sentence[1][1:])]
                #if there are more than two tokens
                if len(sentence)>2:
                    #adds info per token by ennumerating the elements of each token and appending them to the lists of the respective index
                    for tokentuple in sentence[2:]:
                        for i, addition in enumerate(tokentuple[1:]):
                            sortedargumentinfo[i].append(addition)



            #appends each sorted argument list into the list of gold labels per sentence/duplicate sentence
                for argumentlist in sortedargumentinfo:
                    goldarguments.append(argumentlist)


                for tokentuple in sentence:
                    token=tokentuple[0]
                    tokensentence.append(token)


                #after tokens have been appended to each sentence, appends the full sentence as many times as there are verbs
                for i in range(numofverbs):
                    tokenonlysentences.append(tokensentence)
                    sentence_lenght.append(len(tokensentence))

    #finds the index of the verb in a sentence, gets it, appends it to list of verbs, in the correct order
    for sentence, arguments in zip(tokenonlysentences,goldarguments):
        if 'V' in arguments:
            indexofverb=arguments.index('V')
            verb=sentence[indexofverb]
            verbs.append(verb)



    for goldlabellist in goldarguments:
        #remove V from gold labels, with indexing
        goldlabellist[goldlabellist.index("V")] = "_"


    #writing to a file
    rows=[]
    #appends first the tokens and labels, and keeps the index for later appending the verb
    for groupindex, (sentences, goldargs) in enumerate(zip(tokenonlysentences, goldarguments)):
        for token, goldlabel in zip(sentences, goldargs):
            rows.append((groupindex,token, goldlabel))

    with open(outputfile, "w", encoding='utf8',newline="" ) as file:
        writer = csv.writer(file, delimiter="\t")
        for groupindex, token, goldlabel in rows:
            writer.writerow([token, goldlabel])

    with open(outputfile, encoding='utf8',newline="") as file:
        tokenlabel = list(csv.reader(file, delimiter="\t"))

    with open(outputfile, "w", encoding='utf8',newline="") as file:
        writer = csv.writer(file, delimiter="\t")
        for (group_index,token, goldlabel), row in zip(rows, tokenlabel):
            #uses the index found in the rows list for appending the correct verb to each token row
            writer.writerow(row + [verbs[group_index]])

    return verbs, sentence_lenght




In [5]:
def readpreprocessedfile(preprocessedfile):

    tokenonlysentences, goldarguments,verbs = [], [], []

    with open(preprocessedfile, newline="", encoding='utf8') as file:
        reader = csv.reader(file, delimiter="\t")
        for row in reader:
            tokenonlysentences.append(row[0])
            goldarguments.append(row[1])
            verbs.append(row[2])

    return tokenonlysentences, goldarguments,verbs

In [ ]:
import csv



train_sents=extract_tokens_and_arg_info('../data/en_ewt-up-train.conllu')

verbs_train,train_sents_lenght=data_preprocessing(train_sents,'../data/train_preprocessed.tsv')

#verbs are repeated per token per sentence
tokens_preprocessed_train, goldarguments_train, repeated_verbs_train=readpreprocessedfile('../data/train_preprocessed.tsv')


#DEV
dev_sents=extract_tokens_and_arg_info('../data/en_ewt-up-dev.conllu')

#creates preprocessing file and also returns the list of unrepeated verbs
verbs_dev, dev_sents_lenght=data_preprocessing(dev_sents,'../data/dev_preprocessed.tsv')

#verbs are repeated per token per sentence
tokens_preprocessed_dev, goldarguments_dev,repeated_verbs_dev=readpreprocessedfile('../data/dev_preprocessed.tsv')

#TEST
test_sents=extract_tokens_and_arg_info('../data/en_ewt-up-test.conllu')

#creates preprocessing file and also returns the list of unrepeated verbs
verbs_test,test_sents_lenght=data_preprocessing(test_sents,'../data/test_preprocessed.tsv')

#verbs are repeated per token per sentence
tokens_preprocessed_test, goldarguments_test,repeated_verbs_test=readpreprocessedfile('../data/test_preprocessed.tsv')



print(tokens_preprocessed_train[:10],goldarguments_train[:10], repeated_verbs_train[:10])

['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al'] ['_', '_', '_', '_', '_', 'ARG0', '_', 'ARG1', '_', '_'] ['killed', 'killed', 'killed', 'killed', 'killed', 'killed', 'killed', 'killed', 'killed', 'killed']


In [7]:
def adding_special_tokens_to_predicate(tokens_preprocessed, gold_arguments, repeated_verbs, unique_verbs,sentence_length):
    """Adds a marker to the relevant predicate of the sentence, as well as extra labels to the gold arguments lists
    Args.
      outputs of the reading preprocessing file"""


    listofsentenceswithmarkedverb=[]
    listofgoldlabelswithmarkedverb=[]

    sentences = []
    gold_labels=[]

    index = 0

    #divides data in sentences
    for length in sentence_length:
      sentence = tokens_preprocessed[index : index + length]
      sentences.append(sentence)
      goldargsent = gold_arguments[index : index + length]
      gold_labels.append(goldargsent)
      index += length


    #per sentence in the dataset adds the special tokens around the verb and also two extra labels to the respective gold labels list
    for sentence, labels, targetverb in zip(sentences, gold_labels, unique_verbs):
      positionofverb= sentence.index(targetverb)
      #inserts the beginning of predicate marking for the sentence, and a tag for the labels
      sentence.insert(positionofverb,'<v>')
      labels.insert(positionofverb,'_')
      #finds the updated position
      positionofverb= sentence.index(targetverb)
      #inserts the closing predicate marker by using the index plus one
      sentence.insert(positionofverb+1,'</v>')
      labels.insert(positionofverb+1,'_')

      listofsentenceswithmarkedverb.append(sentence)
      listofgoldlabelswithmarkedverb.append(labels)


    return  listofsentenceswithmarkedverb, listofgoldlabelswithmarkedverb

In [8]:
markeddatatrain,markedlabelstrain=adding_special_tokens_to_predicate(tokens_preprocessed_train, goldarguments_train, repeated_verbs_train,verbs_train, train_sents_lenght)
print(markeddatatrain[:5],markedlabelstrain[:5])
print(len(markeddatatrain), len(markedlabelstrain))

print(len(tokens_preprocessed_train), len(goldarguments_train), len(repeated_verbs_train))

print(len(verbs_train))


[['Al', '-', 'Zaman', ':', 'American', 'forces', '<v>', 'killed', '</v>', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.'], ['[', 'This', '<v>', 'killing', '</v>', 'of', 'a', 'respected', 'cleric', 'will', 'be', 'causing', 'us', 'trouble', 'for', 'years', 'to', 'come', '.', ']'], ['[', 'This', 'killing', 'of', 'a', 'respected', 'cleric', 'will', '<v>', 'be', '</v>', 'causing', 'us', 'trouble', 'for', 'years', 'to', 'come', '.', ']'], ['[', 'This', 'killing', 'of', 'a', 'respected', 'cleric', 'will', 'be', '<v>', 'causing', '</v>', 'us', 'trouble', 'for', 'years', 'to', 'come', '.', ']'], ['[', 'This', 'killing', 'of', 'a', 'respected', 'cleric', 'will', 'be', 'causing', 'us', 'trouble', 'for', 'years', 'to', '<v>', 'come', '</v>', '.', ']']] [['_', '_', '_', '_', '_', 'ARG0', '_', '_', '_', 'ARG1', '_', '_', '_', '_', '_', '_', '_', '_', '_', 'ARGM-LOC', '_', '_', '_', 

In [9]:
#DEV
markeddatadev,markedlabelsdev=adding_special_tokens_to_predicate(tokens_preprocessed_dev, goldarguments_dev, repeated_verbs_dev,verbs_dev, dev_sents_lenght)


In [10]:
#TEST
markeddatatest,markedlabelstest=adding_special_tokens_to_predicate(tokens_preprocessed_test, goldarguments_test, repeated_verbs_test,verbs_test, test_sents_lenght)


## Preprocessing the data

In [11]:
model_checkpoint = "bert-base-uncased"

In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

c:\Users\ritav\Desktop\VUthings\advanced_NLP\NLP_code\NLP.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Before we can feed those texts to our model, we need to preprocess them. This is done by a 🤗 Transformers `Tokenizer` which will (as the name indicates) tokenize the inputs (including converting the tokens to their corresponding IDs in the pretrained vocabulary) and put it in a format the model expects, as well as generate the other inputs that model requires.

To do all of this, we instantiate our tokenizer with the `AutoTokenizer.from_pretrained` method, which will ensure:

- we get a tokenizer that corresponds to the model architecture we want to use,
- we download the vocabulary used when pretraining this specific checkpoint.

That vocabulary will be cached, so it's not downloaded again the next time we run the cell.

The following assertion ensures that our tokenizer is a fast tokenizers (backed by Rust) from the 🤗 Tokenizers library. Those fast tokenizers are available for almost all models, and we will need some of the special features they have for our preprocessing.

In [13]:


import transformers
assert isinstance(tokenizer, transformers.PreTrainedTokenizerFast)

We're now ready to write the function that will preprocess our samples. We feed them to the `tokenizer` with the argument `truncation=True` (to truncate texts that are bigger than the maximum size allowed by the model) and `is_split_into_words=True` (as seen above). Then we align the labels with the token ids using the strategy we picked:

In [14]:
label_all_tokens = True



In [15]:
#finding the possible labels
possiblelabels=list(set(goldarguments_train+goldarguments_test))
possiblelabels.sort()


print(possiblelabels)

['ARG0', 'ARG1', 'ARG1-DSP', 'ARG2', 'ARG3', 'ARG4', 'ARG5', 'ARGA', 'ARGM-ADJ', 'ARGM-ADV', 'ARGM-CAU', 'ARGM-COM', 'ARGM-CXN', 'ARGM-DIR', 'ARGM-DIS', 'ARGM-EXT', 'ARGM-GOL', 'ARGM-LOC', 'ARGM-LVB', 'ARGM-MNR', 'ARGM-MOD', 'ARGM-NEG', 'ARGM-PRD', 'ARGM-PRP', 'ARGM-PRR', 'ARGM-REC', 'ARGM-TMP', 'C-ARG0', 'C-ARG1', 'C-ARG1-DSP', 'C-ARG2', 'C-ARG3', 'C-ARG4', 'C-ARGM-ADV', 'C-ARGM-COM', 'C-ARGM-CXN', 'C-ARGM-DIR', 'C-ARGM-EXT', 'C-ARGM-GOL', 'C-ARGM-LOC', 'C-ARGM-MNR', 'C-ARGM-PRP', 'C-ARGM-PRR', 'C-ARGM-TMP', 'C-V', 'R-ARG0', 'R-ARG1', 'R-ARG2', 'R-ARG3', 'R-ARG4', 'R-ARGM-ADJ', 'R-ARGM-ADV', 'R-ARGM-CAU', 'R-ARGM-COM', 'R-ARGM-DIR', 'R-ARGM-GOL', 'R-ARGM-LOC', 'R-ARGM-MNR', 'R-ARGM-TMP', 'V', '_']


In [16]:
import time
label2id = {label: i for i, label in enumerate(possiblelabels)}

def tokenize_and_align_labels(markeddata,markedlabels):
    start = time.time()
    #to debug runtime issues
    print("starting tokenizer...")
    tokenized_inputs = tokenizer(markeddata, truncation=True, is_split_into_words=True)
    print(f"Tokenizer: {time.time() - start:.2f}s")
    labels = []
    for i, label in enumerate(markedlabels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            #changed to be in string form
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(label2id[label[word_idx]] if label_all_tokens else -100)

        labels.append(label_ids)
        #runtime indicators per batch
        if i % 1000 == 0:
          print(f"Processed {i}/{len(markedlabelstrain)} sentences in {time.time() - start:.2f}s")

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [ ]:
tokenized_train=tokenize_and_align_labels(markeddatatrain,markedlabelstrain)
print(tokenized_train[:5])


starting tokenizer...
Tokenizer: 2.50s
Processed 0/40451 sentences in 2.50s
Processed 1000/40451 sentences in 2.50s
Processed 2000/40451 sentences in 2.51s
Processed 3000/40451 sentences in 2.52s
Processed 4000/40451 sentences in 2.53s
Processed 5000/40451 sentences in 2.53s
Processed 6000/40451 sentences in 2.54s
Processed 7000/40451 sentences in 2.54s
Processed 8000/40451 sentences in 2.55s
Processed 9000/40451 sentences in 2.55s
Processed 10000/40451 sentences in 2.56s
Processed 11000/40451 sentences in 2.56s
Processed 12000/40451 sentences in 2.57s
Processed 13000/40451 sentences in 2.57s
Processed 14000/40451 sentences in 2.58s
Processed 15000/40451 sentences in 2.58s
Processed 16000/40451 sentences in 2.59s
Processed 17000/40451 sentences in 2.59s
Processed 18000/40451 sentences in 2.60s
Processed 19000/40451 sentences in 2.61s
Processed 20000/40451 sentences in 2.62s
Processed 21000/40451 sentences in 2.63s
Processed 22000/40451 sentences in 2.64s
Processed 23000/40451 sentences

In [2]:
tokenized_test=tokenize_and_align_labels(markeddatatest,markedlabelstest)
#gets the word ids for later remapping
word_ids = [tokenized_test.word_ids(i) for i in range(len(markeddatatest))]
subtoken_ids = tokenized_test["input_ids"]


NameError: name 'tokenize_and_align_labels' is not defined

## Fine-tuning the model

Now that our data is ready, we can download the pretrained model and fine-tune it. Since all our tasks are about token classification, we use the `AutoModelForTokenClassification` class. Like with the tokenizer, the `from_pretrained` method will download and cache the model for us. The only thing we have to specify is the number of labels for our problem (which we can get from the features, as seen before):

In [19]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer


model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(possiblelabels))




Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2990.28it/s]
BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you exp

The warning is telling us we are throwing away some weights (the `vocab_transform` and `vocab_layer_norm` layers) and randomly initializing some other (the `pre_classifier` and `classifier` layers). This is absolutely normal in this case, because we are removing the head used to pretrain the model on a masked language modeling objective and replacing it with a new head for which we don't have pretrained weights, so the library warns us we should fine-tune this model before using it for inference, which is exactly what we are going to do.

To instantiate a `Trainer`, we will need to define three more things. The most important is the [`TrainingArguments`](https://huggingface.co/transformers/main_classes/trainer.html#transformers.TrainingArguments), which is a class that contains all the attributes to customize the training. It requires one folder name, which will be used to save the checkpoints of the model, and all other arguments are optional:


In [20]:
from transformers import set_seed

# Set random seed
SEED = 42
set_seed(SEED)

model_name = model_checkpoint.split("/")[-1]
args = TrainingArguments(
    f"{model_name}-finetuned-srl",
    eval_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    seed=SEED,
    report_to="none",
)

Here we set the evaluation to be done at the end of each epoch, tweak the learning rate, use the `batch_size` defined at the top of the notebook and customize the number of epochs for training, as well as the weight decay.

The last argument to setup everything so we can push the model to the [Hub](https://huggingface.co/models) regularly during training. Remove it if you didn't follow the installation steps at the top of the notebook. If you want to save your model locally in a name that is different than the name of the repository it will be pushed, or if you want to push your model under an organization and not your name space, use the `hub_model_id` argument to set the repo name (it needs to be the full name, including your namespace: for instance `"sgugger/bert-finetuned-ner"` or `"huggingface/bert-finetuned-ner"`).

The data collator will batch the processed examples together while applying padding to make them all the same size (each pad will be padded to the length of its longest example). There is a data collator for this task in the Transformers library, that not only pads the inputs, but also the labels:

In [21]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

The last thing to define for our `Trainer` is how to compute the metrics from the predictions. Here we will load the [`seqeval`](https://github.com/chakki-works/seqeval) metric (which is commonly used to evaluate results on the CONLL dataset) via the Datasets library.

In [24]:
from datasets import load_dataset
from evaluate import load

In [26]:
metric = load("seqeval")

This metric takes a list of labels for the predictions and references:

In [27]:
labels = [possiblelabels[i] for i in tokenized_train["labels"][0] if i != -100]
metric.compute(predictions=[labels], references=[labels])

c:\Users\ritav\Desktop\VUthings\advanced_NLP\NLP_code\NLP.venv\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: _ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\ritav\Desktop\VUthings\advanced_NLP\NLP_code\NLP.venv\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: ARG0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\ritav\Desktop\VUthings\advanced_NLP\NLP_code\NLP.venv\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: ARG1 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\ritav\Desktop\VUthings\advanced_NLP\NLP_code\NLP.venv\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: ARGM-LOC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


{'LOC': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(1)},
 'RG0': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(1)},
 'RG1': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(1)},
 '_': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(3)},
 'overall_precision': np.float64(1.0),
 'overall_recall': np.float64(1.0),
 'overall_f1': np.float64(1.0),
 'overall_accuracy': 1.0}

So we will need to do a bit of post-processing on our predictions:
- select the predicted index (with the maximum logit) for each token
- convert it to its string label
- ignore everywhere we set a label of -100

The following function does all this post-processing on the result of `Trainer.evaluate` (which is a namedtuple containing predictions and labels) before applying the metric:

In [28]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [possiblelabels[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [possiblelabels[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

making the data into dataset objcets to pass them to the trainer


In [29]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "input_ids": tokenized_train["input_ids"],
    "attention_mask": tokenized_train["attention_mask"],
    "labels": tokenized_train["labels"]
})

test_dataset = Dataset.from_dict({
    "input_ids": tokenized_test["input_ids"],
    "attention_mask": tokenized_test["attention_mask"],
    "labels": tokenized_test["labels"]
})

In [30]:


trainer = Trainer(
    model,
    args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

We can now finetune our model by just calling the `train` method:

In [31]:
trainer.train()

c:\Users\ritav\Desktop\VUthings\advanced_NLP\NLP_code\NLP.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
model.save_pretrained("../models/bert_finetuned_srl_ML")
tokenizer.save_pretrained("../models/bert_finetuned_srl_ML")

I chose to use the majority label for each word, since it is a straightforward solution for handling the subtoken to token mapping

In [1]:

def majority(predictionlist):
  """
  Get the majority label from a list predictions

  :param lst: List of predicted labels
  :return: Majority label
  """
  m = np.mean(predictionlist)
  return 1 if m > 0.5 else 0



def subtoken_to_token_predictions(predictions, word_ids, subtoken_ids):
  """
  Map subtoken predictions to token predictions.

  :param predictions: List of lists of subtoken predictions
  :param word_ids: List of lists of word ids
  :param subtoken_ids: List of lists of subtoken ids
  :return: List of lists of token predictions
  """  
  # Remove ignored index (special tokens)
  predicate_marking_tokens = ["<v>", "</v>"]
  predicate_marking_ids = tokenizer.convert_tokens_to_ids(predicate_marking_tokens)
  normalized_predictions = np.argmax(predictions, axis=2)

  token_predictions = []
  current_word_preds = []

  # loop through the sentences
  for prediction, words, subtokens in zip(normalized_predictions, word_ids, subtoken_ids):
    previous_word_id = None
    sentence_prediction_ids = []
    for pred,word_id,subtoken_id in zip(prediction,words,subtokens):
        #ignores cls tokens and verb marking tokens
        if word_id is None or subtoken_id in predicate_marking_ids:
           continue
        #if the we change words 
        elif word_id != previous_word_id:
            #use majority voting to choose a label on the current word preds
            majority_label = majority(current_word_preds)
            #empties the list for next iteration
            current_word_preds = []
            #appends the prediction for the word to the prediction list
            sentence_prediction_ids.append(majority_label)
        # For the other tokens in a word, we set the label to either the current label or -100, depending on
        # the label_all_tokens flag.
        elif word_id == previous_word_id:
            #apends current word prediction for later decision
            current_word_preds.append(pred)
        else:
            continue

        previous_word_id = word_id

    token_predictions.append(sentence_prediction_ids)

  return token_predictions



To get the precision/recall/f1 computed for each category now that we have finished training, we can apply the same function as before on the result of the `predict` method:

In [ ]:
predictions, labels, _ = trainer.predict(test_dataset)
predictions = np.argmax(predictions, axis=2)

In [ ]:

#map subtokens to tokens using majority voting
token_preds = subtoken_to_token_predictions(predictions, word_ids, subtoken_ids)

#convert predicted integer ids to label strings
true_predictions = [[possiblelabels[p] for p in sentence] for sentence in token_preds]

#gold labels - remove <v> and </v> positions in the gold labels to match
true_labels = [[l for l in sentence if l not in ['<v>', '</v>']] for sentence in markedlabelstest ]

results = metric.compute(predictions=true_predictions, references=true_labels)
print(results)

In [ ]:
#writting predictions to a file

rows = []
for sentence, gold, pred in zip(markeddatatest, true_labels, true_predictions):
    clean_sentence = [t for t in sentence if t not in ['<v>', '</v>']]
    for token, gold_label, pred_label in zip(clean_sentence, gold, pred):
        rows.append((token, gold_label, pred_label))
    #line betwwen sentences
    rows.append(('', '', ''))  

with open('../models/bert_predictions.tsv', 'w', encoding='utf8', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['token', 'gold_label', 'predicted_label']) 
    for row in rows:
        writer.writerow(row)

Ready to use function

In [ ]:
import torch
#using torch to have easier access to token ids in a standalone case

def perform_srl_on_single_sentence(sentence, predicate_indicator, model, tokenizer, possiblelabels):
    """
    Perform Semantic Role Labeling on a standalone sentence given a predicate.
    
    Args:
        sentence (list): Sentence as a list of strings 
        predicate_indicator (list): Binary list indicating predicate position
        model: Finetuned BertForTokenClassification model
        tokenizer: BERT tokenizer
        possiblelabels (list): List of label strings    
    
    Returns:
        dict: token -> predicted label mapping
    """
    #evaluation mode
    model.eval()
    

    #find predicate position and add <v> </v> markers for passing it to bert
    predicate_idx = predicate_indicator.index(1)
    #to not alter the acutual sentence, which will be also later needed
    marked_sentence = sentence.copy() 
    marked_sentence.insert(predicate_idx, '<v>')
    marked_sentence.insert(predicate_idx + 2, '</v>')

    #tokenize the sentence with markers
    inputs = tokenizer(
        marked_sentence,
        is_split_into_words=True,
        truncation=True,
    )

   
    #running the model
    with torch.no_grad():
        outputs = model(**inputs)

    #make it into embedded list structure needed for subtoken_to_token_predictions, and converts it into a numpy array
    predictions = outputs.logits.numpy() 
    #getting ids of words and subwords for the sentence
    word_ids = [inputs.word_ids(batch_index=0)]
    subtoken_ids = inputs['input_ids'].tolist()

    #map subtokens to tokens
    token_preds = subtoken_to_token_predictions(predictions, word_ids, subtoken_ids)

    #converts predictions to readably strings
    token_labels = [possiblelabels[p] for p in token_preds[0]]

    #maps tokens to the the desired label
    result = {token: label for token, label in zip(sentence, token_labels)}
    return result



In [ ]:
#loading the model adn tokenizer
model = AutoModelForTokenClassification.from_pretrained("../models/bert_finetuned_srl_ML")
tokenizer = AutoTokenizer.from_pretrained("../models/bert_finetuned_srl_ML")

onesentencetest=perform_srl_on_single_sentence(['Pia', 'asked', 'Luis', 'to', 'write', 'this', 'sentence','.'], [0,0,0,0,1,0,0,0], model, tokenizer, possiblelabels)

print(onesentencetest)